# 🖼️ ResNet Residual & Bottleneck Blocks (PyTorch)

This notebook implements the architectural blocks that revolutionized deep computer vision: **Residual Blocks** and **Bottleneck Blocks**, introduced in the landmark paper *'Deep Residual Learning for Image Recognition'* (He et al., 2015).

### The Residual Concept
Instead of hoping stacked layers fit a desired underlying mapping $H(x)$, we let these layers fit a residual mapping $F(x) = H(x) - x$. The original mapping is recast into $F(x) + x$, achieved via a shortcut connection (identity mapping).


## 1. Import Dependencies


In [ ]:
import torch
import torch.nn as nn

print(f"PyTorch Version: {torch.__version__}")


## 2. Basic Residual Block (ResNet-18 & ResNet-34)
Composed of two $3\times3$ convolutional layers. If the dimension changes (due to stride), we project the identity shortcut using a $1\times1$ convolution.


In [ ]:
class BasicBlock(nn.Module):
    expansion = 1
    
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        
        self.shortcut = nn.Sequential()
        # If spatial size changes or input channels don't match output channels
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * planes)
            )
            
    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        # Add the identity shortcut mapping
        out += self.shortcut(x)
        out = torch.relu(out)
        return out


## 3. Bottleneck Block (ResNet-50, 101, & 152)
For deeper networks, a three-layer bottleneck structure is used: $1\times1$, $3\times3$, and $1\times1$ convolutions. The $1\times1$ layers are responsible for reducing and restoring dimensions, leaving the $3\times3$ layer a bottleneck with smaller input/output dimensions.


In [ ]:
class BottleneckBlock(nn.Module):
    expansion = 4
    
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, self.expansion * planes, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(self.expansion * planes)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * planes)
            )
            
    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = torch.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        # Add shortcut link
        out += self.shortcut(x)
        out = torch.relu(out)
        return out


## 4. Test the Architectures


In [ ]:
# Generate dummy tensor representing a batch of 2 RGB images of size 64x64
dummy_input = torch.randn(2, 64, 64, 64)

# Instantiate Blocks
basic = BasicBlock(in_planes=64, planes=64)
bottleneck = BottleneckBlock(in_planes=64, planes=16, stride=2) # 16*4 = 64 channels out, strides down

out_basic = basic(dummy_input)
out_bottleneck = bottleneck(dummy_input)

print("Blocks Run Successfully!")
print(f"Input shape:      {dummy_input.shape}")
print(f"Basic output:      {out_basic.shape}")
print(f"Bottleneck output: {out_bottleneck.shape}")
